# Linkages Package: Module-by-Module Exploration

This notebook breaks down each module in the linkages package so you can understand and test each component individually.

## Overview

The pipeline maps company product descriptions to Harmonized System (HS) trade codes using:
1. **Claude** generates search terms
2. **FAISS** finds candidate HS codes via semantic similarity  
3. **GPT** reranks to select the best match

Let's explore each module step by step!

In [ ]:
# Setup: Import everything we'll need
import sys
from pathlib import Path

# Add src to path so we can import linkages
project_root = Path.cwd()
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

# Standard imports
import pandas as pd
import polars as pl
import numpy as np
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("✓ Imports successful!")

## Module 1: `config.py` - Configuration Management

**Purpose**: Loads settings, API keys, and resolves file paths.

**Key Features**:
- Loads API keys from `.env` file
- Provides sensible defaults for all paths
- Centralized configuration dataclass

Let's see how it works:

In [ ]:
from linkages.config import Settings

# Create settings instance
settings = Settings()

print("Configuration loaded!")
print(f"\nProject root: {settings.project_root}")
print(f"Data directory: {settings.data_dir}")
print(f"HS table path: {settings.hs_table_path}")
print(f"Embeddings path: {settings.embeddings_path}")
print(f"Base data path: {settings.base_df_path}")

print(f"\nModel settings:")
print(f"  Embedding model: {settings.embedding_model_name}")
print(f"  Claude model: {settings.claude_model}")
print(f"  GPT model: {settings.gpt_model}")

print(f"\nPipeline parameters:")
print(f"  HS level: {settings.hs_level}")
print(f"  Top K total: {settings.top_k_total}")
print(f"  Top K BERT: {settings.top_k_bert}")
print(f"  Checkpoint every: {settings.checkpoint_every} rows")

## Module 2: `utils.py` - Helper Functions

**Purpose**: Shared utility functions used across modules.

**Key Functions**:
- `extract_text_chunks()`: Extracts text from nested paragraph structures
- `format_table_for_llm()`: Formats HS codes for LLM prompts
- `normalized_embeddings()`: Encodes text and L2-normalizes for cosine similarity

Let's test each one:

In [ ]:
from linkages.utils import extract_text_chunks, format_table_for_llm, normalized_embeddings

# Test extract_text_chunks with different input formats
print("=== Testing extract_text_chunks() ===\n")

# Test 1: List of strings
test1 = ["This is paragraph one.", "This is paragraph two."]
result1 = extract_text_chunks(test1)
print(f"List of strings: {test1}")
print(f"Result: '{result1}'\n")

# Test 2: List of dicts
test2 = [{"text": "First paragraph"}, {"content": "Second paragraph"}]
result2 = extract_text_chunks(test2)
print(f"List of dicts: {test2}")
print(f"Result: '{result2}'\n")

# Test 3: Plain string
test3 = "Just a plain string"
result3 = extract_text_chunks(test3)
print(f"Plain string: '{test3}'")
print(f"Result: '{result3}'\n")

# Test 4: Nested dict
test4 = {"para1": "First", "para2": "Second"}
result4 = extract_text_chunks(test4)
print(f"Dict: {test4}")
print(f"Result: '{result4}'")

In [ ]:
# Test format_table_for_llm
print("=== Testing format_table_for_llm() ===\n")

# Create a sample DataFrame
sample_df = pd.DataFrame({
    'Code': ['0101', '0102', '0103'],
    'Description': ['Live horses', 'Live cattle', 'Live pigs']
})

formatted = format_table_for_llm(sample_df)
print("Sample DataFrame:")
print(sample_df)
print("\nFormatted for LLM:")
for line in formatted:
    print(f"  {line}")

In [ ]:
# Test normalized_embeddings (requires model download - may take a minute)
print("=== Testing normalized_embeddings() ===\n")
print("Loading embedding model (this may take a minute on first run)...")

from sentence_transformers import SentenceTransformer
from linkages.embeddings import load_embedding_model

model = load_embedding_model(settings.embedding_model_name)
print(f"✓ Model loaded: {settings.embedding_model_name}\n")

# Test with sample texts
test_texts = ["semiconductor", "integrated circuit", "electronic component"]
embeddings = normalized_embeddings(test_texts, model)

print(f"Input texts: {test_texts}")
print(f"Embedding shape: {embeddings.shape}")
print(f"Embedding dtype: {embeddings.dtype}")

# Verify normalization (each row should have L2 norm = 1.0)
norms = np.linalg.norm(embeddings, axis=1)
print(f"\nL2 norms (should all be ~1.0): {norms}")
print(f"✓ All normalized: {np.allclose(norms, 1.0)}")

## Module 3: `embeddings.py` - HS Code Embedding Generation

**Purpose**: Generate and load S-BERT embeddings for HS code descriptions.

**Key Functions**:
- `load_embedding_model()`: Load SentenceTransformer model
- `generate_embeddings()`: Generate embeddings for all HS codes (one-time step)
- `load_embeddings()`: Load pre-computed embeddings
- `load_hs_data()`: Load HS code reference table

Let's explore:

In [ ]:
from linkages.embeddings import load_hs_data, load_embeddings

# Load HS data
print("=== Loading HS Code Reference Table ===\n")

if settings.hs_table_path.exists():
    hs_data = load_hs_data(
        settings.hs_table_path,
        sheet_name=settings.hs_sheet,
        level=settings.hs_level
    )
    print(f"✓ Loaded {len(hs_data)} HS{settings.hs_level} codes")
    print(f"\nFirst 5 rows:")
    print(hs_data.head())
    print(f"\nColumns: {list(hs_data.columns)}")
else:
    print(f"✗ HS table not found at: {settings.hs_table_path}")
    print("  Make sure the file exists in data/raw/")

In [ ]:
# Check if embeddings exist
print("=== Checking Embeddings ===\n")

if settings.embeddings_path.exists():
    embeddings = load_embeddings(settings.embeddings_path)
    print(f"✓ Embeddings loaded from: {settings.embeddings_path}")
    print(f"  Shape: {embeddings.shape}")
    print(f"  Dtype: {embeddings.dtype}")
    print(f"  Memory: {embeddings.nbytes / 1024 / 1024:.2f} MB")
    
    # Verify normalization
    sample_norms = np.linalg.norm(embeddings[:10], axis=1)
    print(f"\n  Sample L2 norms (first 10): {sample_norms}")
    print(f"  ✓ Normalized: {np.allclose(sample_norms, 1.0)}")
else:
    print(f"✗ Embeddings not found at: {settings.embeddings_path}")
    print("  Run: uv run python scripts/1_generate_embeddings.py")

## Module 4: `retrieval.py` - FAISS Semantic Search

**Purpose**: Fast semantic search over HS codes using FAISS.

**Key Class**: `HSIndex`
- Builds FAISS index **once** at initialization (major performance improvement!)
- `search()`: Single query search
- `multi_search()`: Multi-query search (original query + Claude terms)

**Performance Note**: The original code rebuilt the FAISS index ~48,000 times. This version builds it once!

Let's test it:

In [ ]:
from linkages.retrieval import HSIndex
from linkages.embeddings import load_embedding_model

print("=== Building FAISS Index ===\n")
print("This builds the index ONCE - the key performance improvement!\n")

# Load embedding model
embedding_model = load_embedding_model(settings.embedding_model_name)

# Build HSIndex (this loads data, embeddings, and builds FAISS index)
if settings.hs_table_path.exists() and settings.embeddings_path.exists():
    hs_index = HSIndex(
        hs_table_path=settings.hs_table_path,
        embeddings_path=settings.embeddings_path,
        embedding_model=embedding_model,
        sheet_name=settings.hs_sheet,
        level=settings.hs_level,
    )
    print("\n✓ HSIndex ready!")
else:
    print("✗ Missing required files. Check HS table and embeddings paths.")

In [ ]:
# Test single query search
print("=== Testing Single Query Search ===\n")

if 'hs_index' in locals():
    query = "semiconductor chip"
    top_k = 5
    
    results = hs_index.search(query, top_k)
    print(f"Query: '{query}'")
    print(f"Top {top_k} results:\n")
    
    for idx, (_, row) in enumerate(results.iterrows(), 1):
        print(f"{idx}. Code {row['Code']}: {row['Description']}")
else:
    print("✗ HSIndex not built. Run previous cell first.")

In [ ]:
# Test multi-query search (simulating Claude-generated terms)
print("=== Testing Multi-Query Search ===\n")

if 'hs_index' in locals():
    query = "quantum processor chip"
    claude_terms = ["semiconductor", "integrated circuit", "electronic component", "computing equipment"]
    
    print(f"Original query: '{query}'")
    print(f"Claude-generated terms: {claude_terms}\n")
    
    results = hs_index.multi_search(
        query=query,
        terms=claude_terms,
        top_k_total=settings.top_k_total,
        top_k_bert=settings.top_k_bert
    )
    
    print(f"Found {len(results)} unique HS codes (after deduplication)\n")
    print("Top results:")
    for idx, (_, row) in enumerate(results.head(10).iterrows(), 1):
        print(f"{idx}. Code {row['Code']}: {row['Description']}")
else:
    print("✗ HSIndex not built. Run previous cell first.")

## Module 5: `rerank.py` - LLM-Based Term Generation & Reranking

**Purpose**: Uses LLMs to generate search terms and rerank candidates.

**Key Functions**:
- `generate_search_terms()`: Claude generates 5-8 search terms
- `rerank_codes()`: GPT selects top 2 HS codes from shortlist

**Note**: These functions make API calls. Make sure your `.env` file has valid API keys!

Let's test them (requires API keys):

In [ ]:
# Test Claude term generation
print("=== Testing Claude Term Generation ===\n")
print("⚠️  This makes an API call - requires valid ANTHROPIC_API_KEY in .env\n")

import anthropic
from linkages.rerank import generate_search_terms

try:
    claude_client = anthropic.Anthropic(
        api_key=settings.anthropic_api_key,
        max_retries=3
    )
    
    # Example product
    query = "AI chip for data centers"
    articles = "Advanced semiconductor technology for neural network acceleration. Quantum computing applications."
    
    # Get HS descriptions for the prompt
    if 'hs_data' in locals():
        hs_descriptions = hs_data["Description"].tolist()[:100]  # Use first 100 for demo
    else:
        hs_descriptions = ["semiconductor", "integrated circuit", "electronic component"]  # Fallback
    
    print(f"Product: '{query}'")
    print(f"Articles: '{articles}'\n")
    print("Calling Claude to generate search terms...\n")
    
    terms = generate_search_terms(
        query=query,
        articles=articles,
        hs_descriptions=hs_descriptions,
        client=claude_client,
        model=settings.claude_model,
    )
    
    print(f"✓ Generated {len(terms)} terms:")
    for i, term in enumerate(terms, 1):
        print(f"  {i}. {term}")
        
except Exception as e:
    print(f"✗ Error: {e}")
    print("  Make sure ANTHROPIC_API_KEY is set in .env")

In [ ]:
# Test GPT reranking
print("=== Testing GPT Reranking ===\n")
print("⚠️  This makes an API call - requires valid OPENAI_API_KEY in .env\n")

from openai import OpenAI
from linkages.rerank import rerank_codes

try:
    gpt_client = OpenAI(api_key=settings.openai_api_key, max_retries=3)
    
    # Create a sample shortlist (simulating FAISS results)
    if 'hs_index' in locals():
        # Get real shortlist from FAISS
        shortlist = hs_index.multi_search(
            query="quantum processor chip",
            terms=["semiconductor", "integrated circuit", "electronic component"],
            top_k_total=15,
            top_k_bert=5
        )
    else:
        # Fallback: create dummy shortlist
        shortlist = pd.DataFrame({
            'Code': ['8541', '8542', '8471', '8514', '8529'],
            'Description': [
                'Semiconductor devices; electronic integrated circuits',
                'Electronic integrated circuits; parts thereof',
                'Automatic data processing machines',
                'Electrical apparatus for line telephony',
                'Parts suitable for use with apparatus of heading 8525 to 8528'
            ]
        })
    
    query = "quantum processor chip"
    articles = "Advanced quantum computing processor for neural network acceleration"
    
    # Load full HS data for description lookup
    if settings.hs_table_path.exists():
        hs_data_full = pl.read_excel(settings.hs_table_path, sheet_name=settings.hs_sheet)
    else:
        hs_data_full = None
    
    print(f"Product: '{query}'")
    print(f"Shortlist has {len(shortlist)} candidates\n")
    print("Calling GPT to rerank and select top 2...\n")
    
    result = rerank_codes(
        shortlist=shortlist,
        query=query,
        articles=articles,
        hs_data=hs_data_full if hs_data_full is not None else pl.DataFrame(),
        client=gpt_client,
        model=settings.gpt_model,
    )
    
    print("✓ Reranking complete!")
    print(f"\nFirst choice:")
    print(f"  Code: {result['code_first']}")
    print(f"  Description: {result['desc_first']}")
    print(f"\nSecond choice:")
    print(f"  Code: {result['code_second']}")
    print(f"  Description: {result['desc_second']}")
    print(f"\nReason: {result['reason']}")
    
except Exception as e:
    print(f"✗ Error: {e}")
    print("  Make sure OPENAI_API_KEY is set in .env")

## Module 6: `pipeline.py` - Full Pipeline Orchestration

**Purpose**: Ties everything together for batch classification.

**Key Functions**:
- `classify_row()`: Classify a single product (all 3 stages)
- `batch_classify()`: Process multiple rows with checkpointing
- `main()`: CLI entry point

Let's test the full pipeline on a single example:

In [ ]:
# Test full pipeline on a single row
print("=== Testing Full Pipeline (Single Row) ===\n")
print("⚠️  This makes API calls to both Claude and GPT\n")

from linkages.pipeline import classify_row
from linkages.embeddings import load_hs_data

try:
    # Setup: Build all required components
    print("Setting up pipeline components...\n")
    
    # 1. Load embedding model
    embedding_model = load_embedding_model(settings.embedding_model_name)
    
    # 2. Build FAISS index
    hs_index = HSIndex(
        hs_table_path=settings.hs_table_path,
        embeddings_path=settings.embeddings_path,
        embedding_model=embedding_model,
        sheet_name=settings.hs_sheet,
        level=settings.hs_level,
    )
    
    # 3. Load HS data (for Claude prompt and GPT lookup)
    hs_data_full = load_hs_data(
        settings.hs_table_path,
        settings.hs_sheet,
        settings.hs_level
    )
    hs_descriptions = hs_data_full["Description"].tolist()
    hs_data_polars = pl.read_excel(settings.hs_table_path, sheet_name=settings.hs_sheet)
    
    # 4. Create API clients
    claude_client = anthropic.Anthropic(
        api_key=settings.anthropic_api_key,
        max_retries=3
    )
    gpt_client = OpenAI(api_key=settings.openai_api_key, max_retries=3)
    
    print("✓ All components ready!\n")
    
    # Create a test row
    test_row = {
        "unique_id": "test_001",
        "name": "AI neural network processor chip",
        "paragraphs": [
            "Advanced semiconductor technology for machine learning applications.",
            "Designed for data center workloads and neural network acceleration.",
            "Quantum computing compatible architecture."
        ]
    }
    
    print(f"Test product: '{test_row['name']}'\n")
    print("Running full classification pipeline...\n")
    
    # Classify!
    result = classify_row(
        row=test_row,
        hs_index=hs_index,
        hs_data=hs_data_polars,
        hs_descriptions=hs_descriptions,
        claude_client=claude_client,
        gpt_client=gpt_client,
        settings=settings,
    )
    
    print("=" * 60)
    print("CLASSIFICATION RESULT")
    print("=" * 60)
    print(f"\nProduct: {test_row['name']}")
    print(f"\nClaude-generated search terms:")
    for i, term in enumerate(result['claude_terms'], 1):
        print(f"  {i}. {term}")
    
    print(f"\nFAISS retrieved {len(result['retrieved_terms'])} candidates")
    print("Top 5 retrieved:")
    for i, term in enumerate(result['retrieved_terms'][:5], 1):
        print(f"  {i}. {term}")
    
    print(f"\n🎯 FINAL CLASSIFICATION:")
    print(f"  First choice: {result['code_first']} - {result['desc_first']}")
    print(f"  Second choice: {result['code_second']} - {result['desc_second']}")
    print(f"\nReasoning: {result['reason']}")
    
except Exception as e:
    print(f"✗ Error: {e}")
    import traceback
    traceback.print_exc()

## Summary: How It All Fits Together

### Pipeline Flow

1. **Configuration** (`config.py`)
   - Loads settings and API keys
   - Resolves all file paths

2. **Data Loading** (`scripts/0_load_data.py`)
   - Loads product data from MongoDB
   - Saves to `base_df.parquet`

3. **Embedding Generation** (`embeddings.py` + `scripts/1_generate_embeddings.py`)
   - One-time: Generate embeddings for all HS codes
   - Uses S-BERT model fine-tuned on trade data

4. **Classification Pipeline** (`pipeline.py`)
   For each product:
   - **Stage 1**: Claude generates 5-8 search terms
   - **Stage 2**: FAISS finds ~25 candidate HS codes (multi-query search)
   - **Stage 3**: GPT reranks and selects top 2 codes

### Key Performance Improvements

- ✅ FAISS index built **once** (not 48k times!)
- ✅ HS data loaded **once** (not per query)
- ✅ Embeddings pre-computed (not regenerated)
- ✅ SDK retry logic (not random sleep)
- ✅ Checkpoint resume (skip already-processed rows)

### Next Steps

- Run full batch: `linkages-classify --limit 10`
- Generate embeddings: `uv run python scripts/1_generate_embeddings.py`
- Load data: `uv run --group scripts python scripts/0_load_data.py`